# 📚 01_数据探索与清洁

> **核心能力**：快速定位数据、条件筛选、去重清洁、类型转换
>
> **开局心法**：拿到表先 `head()`、`info()` 看一眼，然后用 `loc/iloc` 精准定位、用 `.str/.astype` 快速转换。不要用 `for` 循环！

---

## 0️⃣ 基础环境与通用预处理

In [ ]:
import pandas as pd
import numpy as np

# 💡 强迫症必备配置：设置后 Pandas 显示长表时不会省略列名
pd.set_option('display.max_columns', None) 
# pd.set_option('display.max_rows', 100) # (可选) 最多显示行数

In [ ]:
# ============================
# 👉 【代码积木】快速重置与去重
# ============================
def clean_basic(df):
    """一个简单的日常'洗基础数据'小工具"""
    df = df.drop_duplicates()             # 去重
    df = df.dropna(subset=['必须有值的列名']) # 删除特定列为空的行 (注意换列名)
    df = df.reset_index(drop=True)        # 把前面弄乱的 1,2,3,4 序号重新排整齐
    return df

---
## 1️⃣ 快速选数：我到底该怎么查某一行/修改某一个？(`loc` & `iloc`)

> **大白话口诀**：
> *   `loc`：用**名字/标签**找（或者扔个 `True/False` 条件进去找）。
> *   `iloc`：用**数字编号号第几格**找。

In [ ]:
# 假设 df 是我们读进来的表

# 1. 🔍 [查询]：只看 "销量" 前 5 行 (SQL: SELECT 销量 FROM df LIMIT 5)
# display( df[['销量']].head(5) )  

# 2. 🔍 [多条件查询 loc]：找"评价为暴利" 且 "零件为内存条" 的记录
#    注意必须用 (), 且用 '&' (且) 或者是 '|' (或)。
# result = df.loc[ (df['评价'] == '暴利') & (df['硬件类型'] == '内存条') ]

# 3. ✍️ [安全赋值 loc]：当 "内存量" 大于 48 时，给那一行的 "Tier_Level" 列打上 "顶级旗舰" (SQL: CASE WHEN)
# df.loc[df['内存(G)'] > 48, 'Tier_Level'] = '顶级旗舰'
# df.loc[df['内存(G)'] <= 48, 'Tier_Level'] = '常规机型'

# 4. 🔪 [纯数字盲切 iloc]：不管列名叫啥，我就是要前两行、前三列的数据查底看一眼！
# quick_view = df.iloc[0:2, 0:3]

### 🚨 坑点：loc vs iloc 常见错误

#### 坑点 1：loc 多条件必须用括号 + & / |，不能用 and / or

```python
# ❌ 错误写法
result = df.loc[df['价格'] > 100 and df['品牌'] == 'Apple']
# 报错：ValueError: The truth value of a Series is ambiguous

# ✅ 正确写法
result = df.loc[(df['价格'] > 100) & (df['品牌'] == 'Apple')]
```

**原因**：Pandas 的布尔索引是向量化操作，必须用 `&` (逐元素AND) 而不是 `and` (标量AND)。

---

#### 坑点 2：iloc 的切片是左闭右开（[start, end)）

```python
# ❌ 容易的错误理解
df.iloc[0:3]  # 你以为是 0,1,2,3 四行？

# ✅ 实际上
df.iloc[0:3]  # 只有 0,1,2 三行（不包括 3）
```

**记忆技巧**：Python 的所有切片都是左闭右开，和 range() 一样。

---

#### 坑点 3：单行用 loc 返回 Series，多行返回 DataFrame

```python
# 【查一个人的信息】返回 Series
one_person = df.loc[df['姓名'] == '张三']  # 注意！结果是什么？
# 如果只有一个"张三"，返回的还是 DataFrame！（1 行的表，不是 Series）

# 【真的要单个值】如果只想要一个值
single_value = df.loc[df['姓名'] == '张三', '价格'].iloc[0]  # 才是标量（数字）
```

**原因**：`loc` 总是试图保留表的结构。只有当你明确提到某一列时，才会退化成 Series。

---
## 2️⃣ 条件筛选与高级选数

### 使用场景：根据条件快速过滤行、批量修改

#### 【快速操作】按条件筛选特定行

```python
# 找所有"价格 > 5000"的行
expensive = df.loc[df['价格'] > 5000]

# 找所有"品牌为 Apple 或 Dell"的行
specific_brands = df.loc[df['品牌'].isin(['Apple', 'Dell'])]
```

---
## 3️⃣ 数据清洁（去重、删空）

### 何时用
- 数据刚导入时有大量重复行
- 某些关键列有空值，需要删除整行
- 数据经过多次处理，索引变得混乱（需要 reset_index）

In [ ]:
# 1. 🗑️ [去重]：删除完全相同的行
df_clean = df.drop_duplicates()

# 2. 🗑️ [去重进阶]：只按某列去重（比如只按"用户ID"，保留该用户的第一条记录）
df_first_user_record = df.drop_duplicates(subset=['用户ID'], keep='first')

# 3. 🗑️ [删除空值]：删除包含任何 NaN 的行
df_no_na = df.dropna()

# 4. 🗑️ [删除空值进阶]：只删除"订单号"或"客户名"为空的行，其他列有空值无所谓
df_required_filled = df.dropna(subset=['订单号', '客户名'])

# 5. 🔨 [重置索引]：删掉行后，原本的序号 1,3,5,7 变得不连续，用这个重新排整齐
df_clean = df_clean.reset_index(drop=True)
# 🚨 注意：drop=True 表示丢掉旧索引（通常这是你想要的）
#         drop=False 会把旧索引保留成新列（很少用）

### 🚨 坑点：dropna 的陷阱

#### 坑点 1：dropna() 默认删除所有列有任何空值的行

```python
# 【场景】你的表有 1000 行，只因为"备注"列有 100 个空值
df.dropna()  # 结果：删掉了全部 100 行（很粗暴！）

# 【解决】只删那些"关键列"为空的行
df.dropna(subset=['订单号', '金额'])  # 只看这两列，其他列的空值无视
```

---

#### 坑点 2：reset_index 后还有旧索引列

```python
# ❌ 常见错误
df = df.drop_duplicates()
df = df.reset_index()  # 哎呀！多出一列"index"来

# ✅ 正确做法
df = df.reset_index(drop=True)  # 舍去旧索引，只保留 0,1,2,3...
```

---
## 4️⃣ 强制转换类型 (`.astype` & `pd.to_numeric`)

### 何时用
- 从 Excel/CSV 导入的数字被当成了文本
- 日期列需要转成 datetime 格式
- 需要把整数变成浮点数，或反之

In [ ]:
# 1. 📝 [简单转换]：字符串变数字
df['价格'] = df['价格'].astype('float64')
df['年份'] = df['年份'].astype('int32')

# 2. 📝 [安全转换]：如果有脏数据（比如"暂无"），自动变成 NaN
# pd.to_numeric 比 astype 更聪明，遇到无法转换的值会变成 NaN 而不是报错
df['销售额'] = pd.to_numeric(df['销售额'], errors='coerce')
# errors='coerce' 表示：无法转的都变成 NaN
# errors='ignore' 表示：无法转的就保留原值（一般不用）

# 3. 📝 [转成日期]：
df['下单时间'] = pd.to_datetime(df['下单时间'])

# 4. 📝 [转成分类]：节省内存，某列只有有限几个值（比如"性别"只有男女）
df['性别'] = df['性别'].astype('category')

### 🚨 坑点：astype vs to_numeric

#### 坑点 1：astype 遇到脏数据会报错

```python
# ❌ 如果有一个"暂无"混在数字里
df['价格'].astype('float64')  # 报错！

# ✅ 用 to_numeric 就不会
df['价格'] = pd.to_numeric(df['价格'], errors='coerce')
# 结果："暂无"变成 NaN，其他的正常转
```

---

#### 坑点 2：日期转换时格式不对会报错

```python
# ❌ 如果格式混乱（有的是 "2026-03-23"，有的是 "2026/3/23"）
pd.to_datetime(df['日期'])  # 可能报错或结果混乱

# ✅ 指定格式
pd.to_datetime(df['日期'], format='%Y-%m-%d')
```

---
## 5️⃣ 脏字符清理与正则表达式

### 何时用
- 价格列有 `"￥1,200.50"` 这种乱七八糟的格式
- 商品名称有多余空格或特殊符号
- 需要从文本中提取关键信息

In [ ]:
# 1. 🧹 [清理空格]：列名或值两端有空格
df.columns = df.columns.str.strip()  # 列名去空格
df['商品名'] = df['商品名'].str.strip()  # 值去空格

# 2. 🧹 [多余字符替换]：剥离掉 "￥" 和 千分位逗号 ","
# 带 regex=True，允许使用正则表达式
# "[\￥,]" 表示匹配到 "￥" 或者 "," 的地方，全部替换成空串
df['干净价格'] = df['原始价格'].str.replace(r'[\￥,]', '', regex=True)

# 3. 🔍 [检查包含关键词]：判断某个字串是否包含 "RTX" 或 "GDDR"
# 直接用 .str.contains 加管道符 | (代表"或")
keywords = 'RTX|GDDR'
df['含有新显卡'] = df['显卡名'].str.upper().str.contains(keywords, na=False).astype(int)
# na=False：如果值本身是 NaN，返回 False；true='1', false='0'

### 【常用正则表达式速查】

| 正则 | 含义 | 例子 |
|------|------|------|
| `[\￥,]` | 匹配 "￥" 或 "," | 剥离千分号价格 |
| `[^0-9]` | 匹配任何**非数字** | 提取纯数字 |
| `\d+` | 匹配连续数字 | 提取年份、编号 |
| `\s+` | 匹配连续空格 | 多空格变单空格 |
| `^` | 字符串开头 | 以"Pro"开头 |
| `$` | 字符串末尾 | 以".csv"结尾 |
| `abc\|def` | 或 | "Apple" 或 "Dell" |

---

### 🚨 坑点：regex=True 时注意转义

```python
# ❌ 忘记转义反斜杠
df[''].str.replace(r'[\d+]', '')  # 可能出问题

# ✅ 用原始字符串 r'...' 来写正则
df[''].str.replace(r'[\d+]', '', regex=True)  # ✓
```

---
## 6️⃣ 【连招】一行搞定数据清洁

### 读入 → 清洁 → 转换 → 导出

```python
# 读入
df = pd.read_csv('raw.csv', encoding='utf-8')

# 快速清洁三连击
df = df.drop_duplicates()  # 去重
df = df.dropna(subset=['关键列'])  # 删空
df = df.reset_index(drop=True)  # 重排

# 类型转换
df['价格'] = pd.to_numeric(df['价格'], errors='coerce')
df['下单时间'] = pd.to_datetime(df['下单时间'])

# 导出
df.to_csv('clean.csv', index=False, encoding='utf-8-sig')
```